<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Mercosul/Parlamento%20uruguaio/parlamento_uy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Coleta notícias no portal oficial do Parlamento do Uruguai. Se ele estiver indisponível, tenta [notícias oficiais da Câmara de Representantes](https://www.diputados.gub.uy/noticias/). Se os dois portais falharem, coleta [eventos da Biblioteca do Poder Legislativo](https://biblioteca.parlamento.gub.uy/eventos/), com fonte e categoria separadas. **Eventos da Biblioteca não são notícias legislativas.**


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
# Os portais de notícias podem responder HTTP 403/502 em algumas redes.
BASE_URL = "https://parlamento.gub.uy/noticiasyeventos/noticias?field_noticia_fecha_value%5Bmin%5D=2000-07-26&field_noticia_fecha_value%5Bmax%5D=&field_noticia_cuerpo_value=All&body_value="
CHAMBER_NEWS_URL = "https://www.diputados.gub.uy/noticias/"
ALTERNATE_URL = "https://biblioteca.parlamento.gub.uy/eventos/"
MAX_LIBRARY_PAGES = 15  # histórico disponível; reduza para uma coleta rápida
DATABASE_NAME = "parlamento_uy.db"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; FAPESP-Research-Scraper/1.0)"}

print("Fontes: Parlamento UY → Câmara de Representantes → eventos da Biblioteca (cobertura limitada).")


In [ ]:
def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            category TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela prontos")

create_database()

In [ ]:
def insert_article(title, date, category, url, source="Parlamento UY"):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, category, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, category, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [ ]:
def get_total_pages():
    try:
        print("Detectando número de páginas...")
        r = requests.get(BASE_URL, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # Tentativa 1: Busca pelo título 'Ir a la última página'
        # Exemplo HTML: <a href="..." title="Ir a la última página" ...>
        last_page_link = soup.find("a", title="Ir a la última página")

        # Tentativa 2: Busca por texto '»'
        if not last_page_link:
            last_page_link = soup.find("a", string=re.compile(r'»'))

        # Tentativa 3: Busca paginação genérica e pega o maior número
        if not last_page_link:
             print("Paginação 'última' não encontrada, buscando maior número nos links...")
             pagination_links = soup.select("ul.pagination li.page-item a.page-link")
             max_page = 0
             for link in pagination_links:
                 href = link.get('href', '')
                 match = re.search(r'page=(\d+)', href)
                 if match:
                     page_num = int(match.group(1))
                     if page_num > max_page:
                         max_page = page_num
             if max_page > 0:
                 return max_page

        if last_page_link:
            href = last_page_link['href']
            print(f"Link última encontrado: {href}")
            match = re.search(r'page=(\d+)', href)
            if match:
                return int(match.group(1))

        # Fallback manual em caso de falha na detecção automática
        # O usuário informou que espera cerca de 1113 páginas
        raise RuntimeError("Paginação do Parlamento UY não identificada; confira o endereço e os seletores antes da coleta.")

    except Exception as e:
        print(f"Erro ao determinar total de páginas: {e}")
        raise

MAX_PAGES = 1  # páginas recentes; para histórico, use get_total_pages() + 1
total_pages = MAX_PAGES - 1
print(f"Páginas selecionadas: {MAX_PAGES}")

In [ ]:
from urllib.parse import urljoin, urlparse

inserted_count = 0
seen_count = 0
fallback_used = False

for page in range(total_pages + 1):
    url = f"{BASE_URL}&page={page}"
    print(f"Coletando notícias parlamentares: página {page + 1}/{MAX_PAGES}...")
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        fallback_used = True
        print(f"Portal do Parlamento indisponível ({type(exc).__name__}: {exc}).")
        break

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.select("article")
    if not articles:
        fallback_used = True
        print("Portal do Parlamento sem artigos reconhecíveis; tentando a Câmara.")
        break
    for article in articles:
        link_tag = article.select_one("h2.node__title a[href]")
        if not link_tag:
            continue
        title = link_tag.get_text(" ", strip=True)
        link = urljoin("https://parlamento.gub.uy", link_tag["href"])
        if urlparse(link).hostname != "parlamento.gub.uy":
            continue
        date_tag = article.select_one("time")
        date = (date_tag.get("datetime") or date_tag.get_text(" ", strip=True)) if date_tag else ""
        cat_tag = article.select_one(".field--name-field-noticia-cuerpo")
        category = cat_tag.get_text(" ", strip=True) if cat_tag else "Parlamento"
        if title:
            seen_count += 1
            if insert_article(title, date, category, link, "Parlamento UY — notícias"):
                inserted_count += 1
    if page < total_pages:
        time.sleep(0.5)

if seen_count == 0:
    fallback_used = True

if fallback_used:
    print(f"Tentando notícias oficiais da Câmara de Representantes: {CHAMBER_NEWS_URL}")
    chamber_count = 0
    try:
        response = requests.get(CHAMBER_NEWS_URL, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        print(f"Câmara indisponível ({type(exc).__name__}: {exc}).")
    else:
        soup = BeautifulSoup(response.text, "html.parser")
        candidate_links = soup.select('article a[href*="/noticias/"], .post a[href*="/noticias/"], main a[href*="/noticias/"]')
        urls_seen = set()
        for link_tag in candidate_links:
            link = urljoin(response.url, link_tag.get("href", ""))
            parsed = urlparse(link)
            if parsed.hostname not in {"www.diputados.gub.uy", "diputados.gub.uy"} or not parsed.path.startswith("/noticias/"):
                continue
            if parsed.path.rstrip("/") == "/noticias" or link in urls_seen:
                continue
            title = link_tag.get_text(" ", strip=True)
            if len(title) < 12:
                continue
            container = link_tag.find_parent(["article", "div", "li"])
            date_tag = container.select_one("time, .entry-date, .post-date, .date") if container else None
            date = (date_tag.get("datetime") or date_tag.get_text(" ", strip=True)) if date_tag else ""
            urls_seen.add(link)
            chamber_count += 1
            seen_count += 1
            if insert_article(title, date, "Câmara de Representantes", link,
                              "Câmara de Representantes UY — notícias"):
                inserted_count += 1
        if not chamber_count:
            print("Página da Câmara abriu, mas não produziu notícias válidas.")

    if chamber_count == 0:
        response = requests.get(ALTERNATE_URL, headers=HEADERS, timeout=45)
        response.raise_for_status()
        first_soup = BeautifulSoup(response.text, "html.parser")
        page_numbers = [int(n) for a in first_soup.select('a[href*="page="]')
                        for n in re.findall(r"[?&]page=(\d+)", a.get("href", ""))]
        total_library_pages = min(MAX_LIBRARY_PAGES, max(page_numbers, default=1))
        print(f"Biblioteca: coletando até {total_library_pages} páginas do acervo de eventos.")
        library_count = 0
        library_urls = set()
        failed_pages = []
        for library_page in range(1, total_library_pages + 1):
            if library_page == 1:
                page_response, soup = response, first_soup
            else:
                time.sleep(0.2)
                page_url = urljoin(ALTERNATE_URL, f"/eventos?page={library_page}")
                try:
                    page_response = requests.get(page_url, headers=HEADERS, timeout=45)
                    page_response.raise_for_status()
                except requests.RequestException as exc:
                    failed_pages.append(library_page)
                    print(f"Página {library_page} da Biblioteca indisponível: {exc}")
                    continue
                soup = BeautifulSoup(page_response.text, "html.parser")
            cards = soup.select(".evento-container")
            if not cards:
                failed_pages.append(library_page)
                print(f"Página {library_page} da Biblioteca sem eventos reconhecíveis.")
                continue
            for card in cards:
                link_tag = card.select_one('a[href*="/eventos/ver/"]')
                date_tag = card.select_one("h3")
                if not link_tag or not date_tag:
                    continue
                link = urljoin(page_response.url, link_tag.get("href", ""))
                if urlparse(link).hostname != "biblioteca.parlamento.gub.uy" or link in library_urls:
                    continue
                title = link_tag.get_text(" ", strip=True)
                date = date_tag.get_text(" ", strip=True)
                if title and date:
                    library_urls.add(link)
                    seen_count += 1
                    library_count += 1
                    if insert_article(title, date, "Eventos da Biblioteca", link,
                                      "Biblioteca do Poder Legislativo — eventos"):
                        inserted_count += 1
        if not library_count:
            raise RuntimeError("A Biblioteca abriu, mas não produziu registros válidos.")
        print(f"Biblioteca: {library_count} eventos com título, data e URL.")
        if failed_pages:
            print(f"Cobertura parcial: falharam as páginas {failed_pages}.")
        print("Cobertura de eventos da Biblioteca; notícias legislativas indisponíveis nesta execução.")

if not seen_count:
    raise RuntimeError("Nenhum registro válido coletado do Parlamento, da Câmara ou da Biblioteca.")
print(f"📥 Registros vistos: {seen_count}; novos registros inseridos: {inserted_count}")


In [ ]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("SELECT * FROM articles", conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head())

## Panorama e qualidade dos dados

As tabelas e gráficos a seguir descrevem apenas os registros que **esta execução conseguiu coletar**. A coluna `source` distingue notícias parlamentares de eventos da Biblioteca. O número de registros não representa o universo completo de notícias quando algum portal está indisponível.


In [ ]:
import unicodedata
from collections import Counter
from pathlib import Path
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"
if df_db.empty:
    raise RuntimeError("A coleta não trouxe registros para análise.")
df_db = df_db.drop_duplicates(subset=["url"]).copy()

MESES = {
    "enero": 1, "fevereiro": 2, "febrero": 2, "janeiro": 1,
    "marzo": 3, "marco": 3, "março": 3, "abril": 4, "mayo": 5, "maio": 5,
    "junio": 6, "junho": 6, "julio": 7, "julho": 7, "agosto": 8,
    "septiembre": 9, "setiembre": 9, "setembro": 9, "octubre": 10, "outubro": 10,
    "noviembre": 11, "novembro": 11, "diciembre": 12, "dezembro": 12,
}

def normalizar(texto):
    return unicodedata.normalize("NFKD", str(texto).lower()).encode("ascii", "ignore").decode("ascii")

def ler_data(valor):
    texto = normalizar(valor).strip()
    dia_mes = re.search(r"\b(\d{1,2})\s+de\s+([a-z]+)\s+de\s+(\d{4})\b", texto)
    if dia_mes and dia_mes.group(2) in MESES:
        try:
            return pd.Timestamp(int(dia_mes.group(3)), MESES[dia_mes.group(2)], int(dia_mes.group(1)))
        except ValueError:
            return pd.NaT
    iso = re.search(r"\b\d{4}-\d{2}-\d{2}\b", texto)
    if iso:
        return pd.to_datetime(iso.group(), format="%Y-%m-%d", errors="coerce")
    numerica = re.search(r"\b\d{1,2}[/.]\d{1,2}[/.]\d{4}\b", texto)
    if numerica:
        return pd.to_datetime(numerica.group().replace(".", "/"), dayfirst=True, errors="coerce")
    return pd.NaT

df_db["date_clean"] = df_db["date"].apply(ler_data)
df_db["tipo"] = df_db["source"].apply(
    lambda value: "Evento da Biblioteca" if "Biblioteca" in str(value) else "Notícia parlamentar"
)
if "category" not in df_db.columns:
    df_db["category"] = df_db["tipo"].map({
        "Evento da Biblioteca": "Eventos da Biblioteca",
        "Notícia parlamentar": "Notícias legislativas",
    })
datados = df_db.dropna(subset=["date_clean"]).copy()
resumo = pd.DataFrame([
    ("Registros coletados", len(df_db)),
    ("Notícias parlamentares", int((df_db["tipo"] == "Notícia parlamentar").sum())),
    ("Eventos da Biblioteca", int((df_db["tipo"] == "Evento da Biblioteca").sum())),
    ("Datas reconhecidas", len(datados)),
    ("Datas não reconhecidas", len(df_db) - len(datados)),
    ("Primeira data", datados["date_clean"].min().strftime("%d/%m/%Y") if not datados.empty else "—"),
    ("Última data", datados["date_clean"].max().strftime("%d/%m/%Y") if not datados.empty else "—"),
], columns=["Indicador", "Valor"])
display(resumo)
por_fonte = (df_db.groupby(["source", "tipo", "category"]).size()
             .reset_index(name="registros").sort_values("registros", ascending=False))
display(por_fonte)
fig = px.bar(por_fonte, x="source", y="registros", color="tipo", text="registros",
             title="Registros coletados por fonte e tipo",
             labels={"source": "Fonte efetivamente acessada", "registros": "Registros", "tipo": "Tipo"})
fig.update_layout(xaxis_tickangle=-20)
fig.show()
if (df_db["tipo"] == "Notícia parlamentar").sum() == 0:
    print("Atenção: esta execução contém eventos da Biblioteca; os gráficos não medem notícias legislativas.")


## Evolução temporal

As datas são extraídas dos registros disponíveis. Os gráficos mostram a distribuição **do material coletado**, sem inferir que os meses sem registros ficaram sem atividade parlamentar.


In [ ]:
if datados.empty:
    print("Sem datas interpretáveis para construir a série temporal.")
else:
    mensal = (datados.assign(mes=datados["date_clean"].dt.to_period("M"))
              .groupby(["mes", "tipo"]).size().reset_index(name="registros"))
    mensal["mes"] = mensal["mes"].dt.to_timestamp()
    display(mensal.assign(mes=mensal["mes"].dt.strftime("%m/%Y")))
    px.bar(mensal, x="mes", y="registros", color="tipo", barmode="group",
           title="Registros coletados por mês",
           labels={"mes": "Mês", "registros": "Registros", "tipo": "Tipo"}).show()

    anual = (datados.assign(ano=datados["date_clean"].dt.year)
             .groupby(["ano", "tipo"]).size().reset_index(name="registros"))
    display(anual)
    px.bar(anual, x="ano", y="registros", color="tipo", text="registros", barmode="group",
           title="Registros coletados por ano",
           labels={"ano": "Ano", "registros": "Registros", "tipo": "Tipo"}).show()


In [ ]:
if datados.empty:
    print("Sem datas para a linha do tempo.")
else:
    cronologia = datados.sort_values("date_clean").copy()
    cronologia["ordem"] = range(1, len(cronologia) + 1)
    fig = px.scatter(cronologia, x="date_clean", y="ordem", color="tipo",
                     hover_name="title", hover_data={"source": True, "url": True, "ordem": False},
                     title="Cronologia dos registros coletados",
                     labels={"date_clean": "Data", "ordem": "Ordem cronológica", "tipo": "Tipo"})
    fig.show()


## Títulos e temas da pesquisa

As frequências abaixo usam apenas palavras dos **títulos**. Os indicadores temáticos são buscas por termos; uma correspondência não substitui leitura e classificação qualitativa do conteúdo.


In [ ]:
PARADAS = {"para", "como", "sobre", "entre", "com", "uma", "uno", "ante", "este", "esta",
           "esta", "estas", "estos", "desde", "mais", "mas", "por", "del", "los", "las",
           "camara", "representantes", "parlamento", "poder", "legislativo", "biblioteca",
           "uruguay", "uruguai", "2026", "2025", "2024", "2023", "2022", "2021"}
palavras = [p for titulo in df_db["title"].fillna("")
            for p in re.findall(r"[a-z]{4,}", normalizar(titulo)) if p not in PARADAS]
frequencias = pd.DataFrame(Counter(palavras).most_common(18), columns=["palavra", "ocorrencias"])
if frequencias.empty:
    print("Nenhuma palavra suficiente para o gráfico de títulos.")
else:
    display(frequencias)
    px.bar(frequencias.sort_values("ocorrencias"), x="ocorrencias", y="palavra",
           orientation="h", title="Palavras mais frequentes nos títulos coletados",
           labels={"ocorrencias": "Ocorrências", "palavra": "Palavra"}).show()


In [ ]:
TEMAS = {
    "Dados e privacidade": [r"\bdados?\b", r"\bdatos?\b", r"\bprivacidad\b", r"\bprivacidade\b", r"\bproteccion de datos\b"],
    "IA e algoritmos": [r"\binteligencia artificial\b", r"\binteligencia artificial\b", r"\balgoritm", r"\bia\b"],
    "Internet e plataformas": [r"\binternet\b", r"\bplataformas?\b", r"\bredes sociales\b", r"\bdigital\b"],
    "Mercosul e integração": [r"\bmercosur\b", r"\bmercosul\b", r"\bintegracion\b"],
}
titulos_normalizados = df_db["title"].fillna("").map(normalizar)
temas = pd.DataFrame([
    {"tema": tema, "registros": sum(any(re.search(p, titulo) for p in padroes)
                                    for titulo in titulos_normalizados)}
    for tema, padroes in TEMAS.items()
])
display(temas)
if temas["registros"].sum():
    px.bar(temas, x="tema", y="registros", text="registros",
           title="Indicadores temáticos encontrados nos títulos",
           labels={"tema": "Tema", "registros": "Títulos com termos"}).show()
else:
    print("Nenhum desses termos apareceu nos títulos coletados; isto não prova ausência dos temas nos textos completos.")
print("Um título pode entrar em mais de um tema. Os eventos da Biblioteca permanecem identificados como eventos.")


## Tabela consultável e recorte recente

Confira os links e a fonte de cada registro antes de usar os resultados na pesquisa. O CSV salvo na sessão reúne o material mostrado nestas tabelas.


In [ ]:
tabela = df_db.sort_values("date_clean", ascending=False, na_position="last").copy()
tabela["data_interpretada"] = tabela["date_clean"].dt.strftime("%d/%m/%Y").fillna("—")
display(tabela[["data_interpretada", "title", "tipo", "category", "source", "url"]].head(30))

saida = Path("/content/fapesp-data") if Path("/content").exists() else Path.cwd() / "fapesp-data"
saida.mkdir(parents=True, exist_ok=True)
csv_analise = saida / "parlamento_uy_analise.csv"
tabela.drop(columns=["date_clean"]).to_csv(csv_analise, index=False)
print(f"Tabela completa: {len(tabela)} registros em {csv_analise}")


In [ ]:
hoje = pd.Timestamp.now(tz="America/Sao_Paulo").normalize().tz_localize(None)
inicio = hoje - pd.Timedelta(days=6)
semana = datados[datados["date_clean"].between(inicio, hoje)].copy()
print(f"Janela de {inicio:%d/%m/%Y} a {hoje:%d/%m/%Y}: {len(semana)} registros datados na amostra.")
if semana.empty:
    print("Sem registros recentes nesta fonte acessível; não é uma conclusão sobre o portal legislativo bloqueado.")
else:
    display(semana.sort_values("date_clean", ascending=False)[["date", "title", "tipo", "source", "url"]])
    diarios = (semana.groupby(["date_clean", "tipo"]).size().reset_index(name="registros"))
    px.bar(diarios, x="date_clean", y="registros", color="tipo", barmode="group",
           title="Registros datados dos últimos sete dias",
           labels={"date_clean": "Data", "registros": "Registros", "tipo": "Tipo"}).show()


### Limites da interpretação

- Notícias do Parlamento e da Câmara aparecem separadas dos eventos da Biblioteca na coluna `source`.
- Falhas de rede, páginas não visitadas e datas não reconhecidas reduzem a cobertura. Confira o diagnóstico exibido durante a coleta.
- Contagens de títulos e termos servem para exploração inicial; leia os textos antes de tirar conclusões sobre governança digital ou atividade legislativa.
